# 🌾 AgroSuit: crop suitability walkthrough (North Africa)

This notebook walks through the full pipeline behind the dashboard:

1. Download a 10-year **ERA5** climatology and **SoilGrids** topsoil for a site
2. Derive agro-climatic indicators (aridity index, dry months, GDD)
3. Score 13 crops with **FAO fuzzy suitability** and read the limiting factors
4. Scan a whole region on a grid and map the best crop
5. *(Optional)* Sentinel-2 NDVI / SAVI with Google Earth Engine

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # project root

from src.pipeline import analyze_site, scan_region
from src.config import REGIONS
from src.visualization import charts, maps

## 1 · Analyse a single site: Béja, the Tunisian cereal belt

In [ ]:
report = analyze_site(36.73, 9.18, name="Béja", years=10)
report.sources

In [ ]:
{k: report.features[k] for k in ["rain", "et0", "aridity_index", "t_mean", "t_cold", "t_hot", "dry_months", "ph", "clay", "elev"]}

In [ ]:
charts.climograph(report.climatology)

## 2 · Crop ranking and limiting factors

In [ ]:
report.ranking_frame()

In [ ]:
charts.ranking_bar(report.results)

In [ ]:
for r in report.results[:3]:
    print()
    print(f"{r.crop.icon} {r.crop.name}: {r.score:.2f} ({r.code})")
    for hint in r.advice():
        print("  -", hint.replace("**", ""))

## 3 · Rainfed vs irrigated: Sfax olive country

In [ ]:
rainfed = analyze_site(34.74, 10.76, name="Sfax")
irrigated = analyze_site(34.74, 10.76, name="Sfax", irrigated=True)
import pandas as pd
pd.DataFrame({
    "rainfed": {r.crop.name: round(r.score, 2) for r in rainfed.results},
    "irrigated": {r.crop.name: round(r.score, 2) for r in irrigated.results},
}).sort_values("irrigated", ascending=False)

## 4 · Regional grid scan

In [ ]:
df, step = scan_region(REGIONS["Northern Tunisia"], resolution=8, years=3)
df.head()

In [ ]:
maps.scan_map(df, step, layer="best_crop")

In [ ]:
maps.scan_map(df, step, layer="durum_wheat")

In [ ]:
df.to_csv("../outputs/scan_northern_tunisia.csv", index=False)

## 5 · *(Optional)* Sentinel-2 vegetation indices with Earth Engine

Needs `pip install earthengine-api geemap`, `earthengine authenticate` and an `EE_PROJECT`.
The original prototype computed NDVI/SAVI over the whole North African bounding box; here we sample
around the site.

In [ ]:
from src.satellite.sentinel2 import earth_engine_ready, fetch_vegetation_indices
if earth_engine_ready():
    print(fetch_vegetation_indices(36.73, 9.18, year=2025))
else:
    print("Earth Engine not configured: skipping.")